In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/model-dataset/Model_dataset.csv


In [2]:
!pip -q install --no-cache-dir --force-reinstall \
  "numpy==2.1.3" \
  "scipy==1.14.1" \
  "scikit-learn==1.6.1" \
  "pyarrow==15.0.2" \
  "datasets==2.21.0" \
  "transformers==4.47.1" \
  "accelerate==1.2.1" \
  "evaluate==0.4.3" \
  "iterative-stratification==0.1.9" \
  "peft==0.13.2"

print("✅ Install done. NOW: Runtime → Restart session")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 285.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 216.3 MB/s eta 0:00:00
ERROR: Cannot install numpy==2.1.3, pyarrow==15.0.2, scikit-learn==1.6.1 and scipy==1.14.1 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
✅ Install done. NOW: Runtime → Restart session


In [3]:
!pip -q install iterative-stratification==0.1.9


In [4]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
print("✅ iterstrat OK")


✅ iterstrat OK


In [5]:
import os, re, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import f1_score, precision_score, recall_score
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed
)

import pyarrow, datasets, transformers, sklearn, scipy

print("✅ Versions:")
print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("pyarrow:", pyarrow.__version__)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("✅ Environment OK")



2026-01-14 14:10:26.888476: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768399826.911164     215 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768399826.917922     215 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768399826.936925     215 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768399826.936952     215 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768399826.936955     215 computation_placer.cc:177] computation placer alr

✅ Versions:
numpy: 2.0.2
scipy: 1.15.3
sklearn: 1.6.1
pyarrow: 22.0.0
datasets: 4.4.1
transformers: 4.57.1
torch: 2.8.0+cu126
✅ Environment OK


In [6]:
# ---------------------------
# ✅ Config
# ---------------------------
SEED = 42
set_seed(SEED)

TEXT_COL = "Data"
LABEL_COLS = ["Love", "Joy", "Anger", "Surprise", "Sadness", "Fear", "Hate"]

DATA_PATH = "/kaggle/input/model-dataset/Model_dataset.csv"

# P100 safe defaults (good quality + not too slow)
MAX_LEN = 160
EPOCHS  = 10
LR      = 2e-5
WD      = 0.01
WARMUP  = 0.10

# ✅ 3-model ensemble
MODELS = [
    "xlm-roberta-base",
    "microsoft/mdeberta-v3-base",
    "xlm-roberta-large",     # ✅ added
]

OUTDIR = "/kaggle/working/ens3_focal_banglish"
os.makedirs(OUTDIR, exist_ok=True)

# Model-specific safe batches (xlmr-large needs smaller batch)
MODEL_BATCH = {
    "xlm-roberta-base": 16,
    "microsoft/mdeberta-v3-base": 16,
    "xlm-roberta-large": 4,      # ✅ P100 safe
}

MODEL_ACCUM = {
    "xlm-roberta-base": 2,       # effective batch 32
    "microsoft/mdeberta-v3-base": 2,
    "xlm-roberta-large": 8,      # effective batch 32 (4*8)
}


In [7]:
import re

# ✅ Text cleaning function (Banglish-friendly)
_url_re = re.compile(r"(http\S+|www\.\S+)")
_user_re = re.compile(r"@\w+")
_multi_space = re.compile(r"\s+")

def clean_text(text):
    text = str(text)
    text = text.lower().strip()
    text = _url_re.sub(" ", text)
    text = _user_re.sub(" ", text)
    # reduce repeated characters: "cooool" -> "cool"
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = _multi_space.sub(" ", text)
    return text


In [8]:

# ---------------------------
# ✅ Load Data
# ---------------------------
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

missing = [c for c in [TEXT_COL] + LABEL_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}\nFound: {df.columns.tolist()}")

df = df[[TEXT_COL] + LABEL_COLS].copy()
df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("").map(clean_text)

for c in LABEL_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

df = df[df[TEXT_COL].str.strip().astype(bool)].reset_index(drop=True)

print("✅ Samples:", len(df))

✅ Samples: 27809


In [9]:
# ---------------------------
# ✅ Multilabel Stratified Split (70/15/15)
# ---------------------------
y = df[LABEL_COLS].values

msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(msss1.split(df, y))

train_df = df.iloc[train_idx].reset_index(drop=True)
temp_df  = df.iloc[temp_idx].reset_index(drop=True)

y_temp = temp_df[LABEL_COLS].values
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(msss2.split(temp_df, y_temp))

val_df  = temp_df.iloc[val_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)

print(f"✅ Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


✅ Train: 19466 | Val: 4171 | Test: 4172


In [10]:
# ---------------------------
# ✅ POS_WEIGHT
# ---------------------------
train_y = train_df[LABEL_COLS].values.astype(np.float32)
pos = train_y.sum(axis=0)
neg = (train_y.shape[0] - pos)
pos_weight = (neg / (pos + 1e-6))
pos_weight_t = torch.tensor(pos_weight, dtype=torch.float32)
print("✅ pos_weight:", dict(zip(LABEL_COLS, np.round(pos_weight, 2))))

✅ pos_weight: {'Love': np.float32(5.42), 'Joy': np.float32(5.42), 'Anger': np.float32(5.42), 'Surprise': np.float32(5.42), 'Sadness': np.float32(5.42), 'Fear': np.float32(5.42), 'Hate': np.float32(5.42)}


In [11]:

# ---------------------------
# ✅ Metrics helpers
# ---------------------------
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics_default(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid(logits)
    preds = (probs >= 0.5).astype(int)

    return {
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "micro_precision": precision_score(labels, preds, average="micro", zero_division=0),
        "micro_recall": recall_score(labels, preds, average="micro", zero_division=0),
        "subset_accuracy": (preds == labels).all(axis=1).mean(),
        "label_accuracy": (preds == labels).mean(),
    }

def find_best_thresholds_from_probs(probs, y_true, label_cols):
    best_thr = []
    for i in range(y_true.shape[1]):
        best_t, best_f1 = 0.5, -1
        for t in np.linspace(0.20, 0.80, 25):
            y_pred = (probs[:, i] >= t).astype(int)
            f1 = f1_score(y_true[:, i], y_pred, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = float(t)
        best_thr.append(best_t)
    best_thr = np.array(best_thr, dtype=np.float32)
    print("✅ Best thresholds:", dict(zip(label_cols, np.round(best_thr, 3))))
    return best_thr

def evaluate_probs_with_thresholds(probs, y_true, thresholds):
    y_pred = (probs >= thresholds.reshape(1, -1)).astype(int)
    return {
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "subset_accuracy": (y_pred == y_true).all(axis=1).mean(),
        "label_accuracy": (y_pred == y_true).mean(),
    }

In [12]:
# ---------------------------
# ✅ Focal Loss + Custom Trainer (FIXED: device + fp16 safe)
# ---------------------------
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.register_buffer("pos_weight", pos_weight)

    def forward(self, logits, targets):
        # ✅ move pos_weight to same device + dtype as logits (fix cuda/cpu error)
        pos_w = None
        if self.pos_weight is not None:
            pos_w = self.pos_weight.to(device=logits.device, dtype=logits.dtype)

        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction="none", pos_weight=pos_w
        )

        p = torch.sigmoid(logits)
        pt = p * targets + (1 - p) * (1 - targets)
        focal = (1 - pt).pow(self.gamma) * bce
        return focal.mean()


class MultilabelFocalTrainer(Trainer):
    def __init__(self, *args, gamma=2.0, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal = FocalLoss(gamma=gamma, pos_weight=pos_weight)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        # ✅ ensure labels are float and on correct device handled by Trainer
        labels = labels.to(dtype=torch.float32)

        outputs = model(**inputs)
        loss = self.focal(outputs.logits, labels)

        return (loss, outputs) if return_outputs else loss


In [13]:
# ---------------------------
# ✅ HF datasets builder
# ---------------------------
def make_hf_ds(tokenizer, train_df, val_df, test_df):
    def to_ds(pdf):
        return Dataset.from_pandas(pdf, preserve_index=False)

    train_ds = to_ds(train_df)
    val_ds   = to_ds(val_df)
    test_ds  = to_ds(test_df)

    def tokenize_fn(batch):
        return tokenizer(batch[TEXT_COL], truncation=True, max_length=MAX_LEN)

    train_ds = train_ds.map(tokenize_fn, batched=True)
    val_ds   = val_ds.map(tokenize_fn, batched=True)
    test_ds  = test_ds.map(tokenize_fn, batched=True)

    def add_labels(batch):
        labels = np.stack([batch[c] for c in LABEL_COLS], axis=1).astype(np.float32)
        batch["labels"] = labels
        return batch

    train_ds = train_ds.map(add_labels, batched=True)
    val_ds   = val_ds.map(add_labels, batched=True)
    test_ds  = test_ds.map(add_labels, batched=True)

    keep = ["input_ids", "attention_mask", "labels"]
    train_ds.set_format(type="torch", columns=keep)
    val_ds.set_format(type="torch", columns=keep)
    test_ds.set_format(type="torch", columns=keep)
    return train_ds, val_ds, test_ds


In [14]:
# ---------------------------
# ✅ Train one model (FULL UPDATED + SAFE)
#   - Fix: DeBERTa double-backward (disable grad checkpointing)
#   - Fix: PytorchStreamWriter file write failed (disable checkpoint + optimizer save)
# ---------------------------
def train_one_model(model_name):
    print("\n" + "="*80)
    print("🚀 Training:", model_name)
    print("="*80)

    # 🔒 Model-specific MAX_LEN (set BEFORE dataset tokenization)
    global MAX_LEN
    if model_name == "xlm-roberta-large":
        MAX_LEN = 128
    else:
        MAX_LEN = 160

    # tokenizer + datasets
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    train_ds, val_ds, test_ds = make_hf_ds(tokenizer, train_df, val_df, test_df)
    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # config + model
    config = AutoConfig.from_pretrained(model_name)
    config.num_labels = len(LABEL_COLS)
    config.problem_type = "multi_label_classification"

    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)

    # ✅ FIX: DeBERTa + custom loss + grad checkpointing => double backward bug
    is_deberta = "deberta" in model_name.lower()
    if not is_deberta:
        model.gradient_checkpointing_enable()

    # ✅ recommended for stability/memory
    model.config.use_cache = False

    # batches
    batch = MODEL_BATCH[model_name]
    accum = MODEL_ACCUM[model_name]

    out_dir = f"{OUTDIR}/{model_name.replace('/','_')}"

    # ✅ IMPORTANT FIX for "file write failed":
    # - don't save checkpoints during training
    # - don't save optimizer/scaler/scheduler states
    args = TrainingArguments(
        output_dir=out_dir,
        seed=SEED,

        num_train_epochs=EPOCHS,
        learning_rate=LR,
        weight_decay=WD,
        warmup_ratio=WARMUP,
        lr_scheduler_type="cosine",

        per_device_train_batch_size=batch,
        per_device_eval_batch_size=64,
        gradient_accumulation_steps=accum,

        fp16=True,

        gradient_checkpointing=(False if is_deberta else True),

        logging_steps=200,
        eval_strategy="epoch",

        save_strategy="no",          # ✅ NO checkpoint saving (prevents disk crash)
        save_total_limit=1,
        save_only_model=True,        # ✅ don't save optimizer/scaler
        load_best_model_at_end=False,# ✅ requires checkpoints; we disable

        
        metric_for_best_model="micro_f1",
        greater_is_better=True,
        report_to="none"
    )

    trainer = MultilabelFocalTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        compute_metrics=compute_metrics_default,
        callbacks=[],
        gamma=2.0,
        pos_weight=pos_weight_t
    )

    # Train
    trainer.train()

    # Predict probs
    val_probs  = sigmoid(trainer.predict(val_ds).predictions)
    test_probs = sigmoid(trainer.predict(test_ds).predictions)

    # Save final model/tokenizer (small + safe)
    save_dir = f"/kaggle/working/saved_{model_name.replace('/','_')}"
    os.makedirs(save_dir, exist_ok=True)
    trainer.save_model(save_dir)
    tokenizer.save_pretrained(save_dir)

    print("✅ Saved:", save_dir)

    return val_probs, test_probs


In [15]:
import shutil, os
shutil.rmtree(OUTDIR, ignore_errors=True)
os.makedirs(OUTDIR, exist_ok=True)


In [16]:
# ---------------------------
# ✅ Train all models + ensemble
# ---------------------------
val_probs_list, test_probs_list = [], []

for m in MODELS:
    vp, tp = train_one_model(m)
    val_probs_list.append(vp)
    test_probs_list.append(tp)

ens_val_probs  = np.mean(np.stack(val_probs_list, axis=0), axis=0)
ens_test_probs = np.mean(np.stack(test_probs_list, axis=0), axis=0)

y_val  = val_df[LABEL_COLS].values.astype(int)
y_test = test_df[LABEL_COLS].values.astype(int)

best_thr = find_best_thresholds_from_probs(ens_val_probs, y_val, LABEL_COLS)

print("\n✅ VAL (Ensemble + threshold) metrics:")
val_metrics = evaluate_probs_with_thresholds(ens_val_probs, y_val, best_thr)
for k,v in val_metrics.items():
    print(f"{k}: {v:.4f}")

print("\n✅ TEST (Ensemble + threshold) metrics:")
test_metrics = evaluate_probs_with_thresholds(ens_test_probs, y_test, best_thr)
for k,v in test_metrics.items():
    print(f"{k}: {v:.4f}")

final_dir = "/kaggle/working/final_ensemble3"
os.makedirs(final_dir, exist_ok=True)
np.save(os.path.join(final_dir, "best_thresholds.npy"), best_thr)
with open(os.path.join(final_dir, "models_used.txt"), "w") as f:
    f.write("\n".join(MODELS))

print("\n✅ Final ensemble saved to:", final_dir)
print("✅ Files:", os.listdir(final_dir))


🚀 Training: xlm-roberta-base


Map:   0%|          | 0/19466 [00:00<?, ? examples/s]

Map:   0%|          | 0/4171 [00:00<?, ? examples/s]

Map:   0%|          | 0/4172 [00:00<?, ? examples/s]

Map:   0%|          | 0/19466 [00:00<?, ? examples/s]

Map:   0%|          | 0/4171 [00:00<?, ? examples/s]

... (24 outputs left out)

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Micro Precision,Micro Recall,Subset Accuracy,Label Accuracy
1,0.230100,0.206979,0.505197,0.518667,0.378441,0.759630,0.121074,0.768469
2,0.188200,0.186132,0.542393,0.546503,0.406363,0.815320,0.142652,0.785937
3,0.162300,0.171752,0.590174,0.601172,0.463608,0.811798,0.263965,0.824571
4,0.138500,0.167792,0.625506,0.630758,0.500396,0.834030,0.345720,0.844607
5,0.117300,0.174910,0.650309,0.655032,0.542802,0.810918,0.413090,0.864301
6,0.093400,0.193436,0.656584,0.660953,0.550993,0.812239,0.434428,0.867795
7,0.072900,0.214807,0.676658,0.681223,0.583626,0.804975,0.486454,0.880296
8,0.058100,0.246210,0.686627,0.690920,0.604215,0.795069,0.513066,0.887077
9,0.049100,0.263611,0.691043,0.694255,0.618990,0.782082,0.534884,0.891187
10,0.045800,0.268721,0.692864,0.696345,0.623176,0.780101,0.540877,0.892386


✅ Saved: /kaggle/working/saved_xlm-roberta-large
✅ Best thresholds: {'Love': np.float32(0.65), 'Joy': np.float32(0.525), 'Anger': np.float32(0.575), 'Surprise': np.float32(0.675), 'Sadness': np.float32(0.575), 'Fear': np.float32(0.65), 'Hate': np.float32(0.525)}

✅ VAL (Ensemble + threshold) metrics:
micro_f1: 0.7074
macro_f1: 0.7120
micro_precision: 0.6918
micro_recall: 0.7238
subset_accuracy: 0.5764
label_accuracy: 0.9068

✅ TEST (Ensemble + threshold) metrics:
micro_f1: 0.7064
macro_f1: 0.7116
micro_precision: 0.6848
micro_recall: 0.7295
subset_accuracy: 0.5698
label_accuracy: 0.9055

✅ Final ensemble saved to: /kaggle/working/final_ensemble3
✅ Files: ['models_used.txt', 'best_thresholds.npy']


In [17]:
# ===========================
# ✅ Overall (non-subset) Accuracy (Multi-label)
# ===========================
# NOTE:
# - subset_accuracy = exact match (all labels correct for a sample)
# - overall accuracy (label-wise / Hamming accuracy) = correct labels ratio over all labels
# - Jaccard accuracy = per-sample overlap / union (soft "overall" sample score)

# Build hard predictions using the tuned thresholds (already computed above)
y_pred_test = (ens_test_probs >= best_thr.reshape(1, -1)).astype(int)

# ✅ Overall accuracy across all labels (same idea as "label_accuracy")
overall_label_accuracy = (y_pred_test == y_test).mean()

# ✅ Hamming loss (lower is better)
hamming_loss = 1.0 - overall_label_accuracy

# ✅ Jaccard (IoU) accuracy per sample (higher is better)
inter = (y_pred_test & y_test).sum(axis=1)
union = (y_pred_test | y_test).sum(axis=1)
jaccard = np.where(union == 0, 1.0, inter / union).mean()

# ===========================
# ✅ Per-label accuracy (ADDED)
# ===========================
per_label_acc = (y_pred_test == y_test).mean(axis=0)

print("\n📌 Extra accuracy metrics (non-subset):")
print(f"overall_label_accuracy (Hamming accuracy): {overall_label_accuracy:.4f}")
print(f"hamming_loss: {hamming_loss:.4f}")
print(f"jaccard_accuracy (sample IoU): {jaccard:.4f}")

print("\n📌 Per-label accuracy:")
for lbl, acc in zip(LABEL_COLS, per_label_acc):
    print(f"{lbl:>9s}: {acc:.4f}")



📌 Extra accuracy metrics (non-subset):
overall_label_accuracy (Hamming accuracy): 0.9055
hamming_loss: 0.0945
jaccard_accuracy (sample IoU): 0.6554

📌 Per-label accuracy:
     Love: 0.9214
      Joy: 0.8718
    Anger: 0.8665
 Surprise: 0.9427
  Sadness: 0.8818
     Fear: 0.9564
     Hate: 0.8981


/tmp/ipykernel_215/880295389.py:21: RuntimeWarning: invalid value encountered in divide
  jaccard = np.where(union == 0, 1.0, inter / union).mean()
